## 게이트가 추가된 RNN

* RNN은 성능이 좋지 못함  
=> 장기 의존 관계를 잘 학습할 수 없음
* 요즘에는 LSTM, GRU 계층을 주로 사용
* LSTM, GRU에는 게이트라는 구조가 더해져 있음  
=> 시계열 데이터의 장기 의존 관계를 학습할 수 있음


### RNN의 문제점
* 시계열 데이터의 장기 의존 관계를 학습하기 어려움  
    * BPTT에서 기울기 소실 또는 기울기 폭발이 일어나기 때문
        * 기울기 소실: 역전파의 기울기 값이 점점 작아지다가 사라지는 현상
        * 기울기 폭발: 기울기 값이 매우 큰 수가 되는 현상  
        => 두 경우 모두 학습이 제대로 이뤄지지 않음

#### RNN 복습
* RNN 계층은 순환 경로를 가짐
* RNN 계층은 시계열 데이터인 $x_t$를 입력하면 $h_t$를 출력
    * $h_t$: RNN 계층의 은닉 상태라고 하여 과거 정보를 저장

#### 기울기 소실 또는 기울기 폭발
* BPTT로 학습
* RNN 계층이 과거 방향으로 의미 있는 기울기를 전달함으로써 시간 방향의 의존 관계를 학습
* 기울기는 학습해야 할 의미가 있는 정보가 들어 있고, 그것을 과거로 전달함으로써 장기 의존 관계를 학습
* 하지만 기울기가 중간에 사그라들면 가중치 매개변수는 전혀 갱신되지 않게 됨  
    -> 장기 의존 관계를 학습할 수 없게 됨  
    => 기울기가 작아지거나 커질 수 있으며 대부분 둘 중 하나

#### 기울기 소실과 기울기 폭발의 원인
* RNN 계층에서의 시간 방향 기울기 전파에만 주목
* 길이가 T인 시계열 데이터를 가정하여 T번째 정답 레이블로부터 전해지는 기울기가 어떻게 변하는지 보면
    * 역전파로 전해지는 기울기는 차례로 tanh, +, MatMul 연산을 통과

* tanh를 미분 시 값은 1.0 이하잉고 x가 0으로부터 멀어질수록 작아짐  
    -> 역전파에서는 기울기가 tanh 노드를 지날 때마다 값이 작아짐  
    => T번 통과하면 기울기도 T번 반복해서 작아지게 됨
* ReLU로 바꾸면 기울기 소실을 줄일 수 있음

* MatMul에서는 $dh$라는 기울기가 흘러온다고 가정, 역전파는 $dhW_h^T$라는 행렬 곱으로 기울기를 계산
* 매번 같은 가중치인 $W_h$가 사용  
    -> 기울기의 크기는 시간에 비례해 지수적으로 증가  
    => 기울기 폭발
* 행렬 $W_h$가 1보다 크면 지수적으로 증가, 1보다 작으면 지수적으로 감소

* $W_h$가 스칼라가 아니라 행렬이라면?
    * 행렬의 특잇값이 척도가 됨
        * 행렬의 특잇값: 데이터가 얼마나 퍼져 있는가
    * 특잇값이 1보다 큰지 여부를 보면 기울기 크기가 어떻게 변할지 예측 가능

#### 기울기 폭발 대책
* 기울기 클리핑
$$
\text{if } \|\mathbf{\hat g}\| \ge \text{threshold : } \\
\mathbf{\hat g} = \frac{\text{threshold}}{\| \mathbf{\hat g} \|}
$$
* $\mathbf{\hat g}$: 신경망에서 사용되는 모든 매개변수의 기울기를 하나로 모은 것
* 기울기의 L2 노름($\| \mathbf{\hat g} \|$)이 문턱값을 초과하면 두 번째 줄의 수식과 같이 기울기를 수정함}

In [4]:
import numpy as np

dW1 = np.random.rand(3, 3) * 10
dW2 = np.random.rand(3, 3) * 10
grads = [dW1, dW2]
max_norm = 5.0
total_norm = 0

for grad in grads:
    total_norm += np.sum(grad ** 2)
total_norm = np.sqrt(total_norm)

rate = max_norm / (total_norm + 1e-6)
if rate < 1:
    for grad in grads:
        grad *= rate

### 기울기 소실과 LSTM
* RNN 학습에서는 기울기 소실도 큰 문제  
    => 게이트가 추가된 RNN
* 게이트가 추가된 RNN에는 LSTM과 GRU가 있으

#### LSTM의 인터페이스
* $\tanh(h_{t-1}W_h + x_t W_x + b$) 계산을 tanh라는 직사각형 노드 하나로 여김
    * 행렬 곱과 편향의 합, tanh 함수에 의한 변환이 모두 포함

* LSTM의 계층의 인터페이스에는 C라는 경록 있음
    * C: 기억 셀이며 LSTM 전용의 기억 메커니즘
* 기억 셀의 특징은 데이터를 LSTM 계층 내에서만 주고 받음
* LSTM 계층 내에서만 완결, 다른 계층으로는 출력하지 않음
* LSTM의 은닉 상태 h는 RNN과 마찬가지로 다른 계층으로 출력

#### LSTM 계층 조립하기
* LSTM에는 기억 셀 $c_t$가 있음
* $c_t$에는 시각 t에서의 LSTM의 기억이 저장돼 있는데, 과거로부터 시각 t까지에 필요한 모든 정보가 저장돼 있다고 가정
* 필요한 정보를 모두 간직한 기억을 바탕으로 외부 계층에 은닉 상태 $h_t$를 출력
* 출력하는 $h_t$는 기억 셀의 값을 tanh 함수로 변환한 값

* 현재의 기억 셀 $c_t$는 3개의 입력 $(C_{t-1}, h_{t-1}, x_t)$으로부터 계산을 수행해 구할 수 있음
* 핵심은 갱신된 $c_t$를 사용해 은닉 상태 $h_t$를 계산
    * $h_t = \tanh(c_t)$
    * $c_t$의 각 요소에 $\tanh$ 함수를 적용

* 게이트는 데이터의 흐름을 제어
* LSTM에서 사용하는 게이트는 열기, 닫기에 더불어 어느 정도 열지 조절할 수 있음
* 어느 정도를 열림 상태라 부름
* 열림 상태는 0.0~1.0 사이의 실수로 나타냄
* 게이트를 얼마나 열 것인가도 자동으로 학습
* 시그모이드 함수를 사용해 열림 상태를 도출

#### output 게이트
* output 게이트: 다음 은닉 상태 $h_t$의 출력을 담당하는 게이트
* output 게이트의 열림 상태는 입력 $x_t$와 $h_{t-1}$로부터 구함
$$
o = \sigma(x_t W_x^{(o)} + h_{t-1} W_h^{(o)} + b^{(o)})
$$
* 입력 $x_t$에는 가중치 $W_x^{(0)}$가, 이전 시각의 은닉 상태 $h_{t-1}$에는 가중치 $W_h^{(0)}$가 붙어있음
* 행렬들의 곱과 편향을 더한 다음시그모이드 함수를 거쳐 출력게이트의 출력 $o$를 구함
* $o$와 $\tanh(c_t)$의 원소별 곱을 $h_t$로 출력

* output 게이트에서 수행하는 계산을 $\sigma$
* $\sigma$의 출력을 $o$
* $h_t$는 $o$와 $\tanh(c_t)$의 곱으로 계산$(\odot)$
$$
h_t = o \odot \tanh(c_t)
$$

#### forget 게이트
* 기억 셀에 무엇을 잊을까를 명확하게 지시하는 것도 게이트로 해결
* $c_{t-1}$에서 불필요한 기억을 잊게 해주는 게이트

$$
f = \sigma(x_t W_x^{(f)} + h_{t-1} W_h^{(f)} + b^{(f)})
$$
* f와 이전 기억 셀인 $c_{t-1}$과의 원소별 곱을 계산하여 구함
$$
c_t = f \odot c_{t-1}
$$

#### 새로운 기억 셀
* forget 게이트를 거치면서 이전 시각의 기억 셀로부터 잊어야 할 기억이 삭제됨
* 새로 기억해야 할 정보를 기억 셀에 추가해야 함  
=> $\tanh$ 노드를 추가
* $\tanh$ 노드가 계산한 결과가 이전 시각의 기억 셀 $c_{t-1}$에 더해짐
    * $\tanh$ 노드는 게이트가 아니며 새로운 정보를 기억 셀에 추가하는 것이 목적
* $\tanh$ 노드에서 수행하는 계산
$$
g = \tanh(x_t W_x^{(g)} + h_{t-1} W_h^{(g)} + b^{(g)})
$$

#### input 게이트
* g에 게이트를 하나 추가 -> input 게이트
* input 게이트는 g의 각 원소가 추가되는 정보러써의 가치가 얼마나 큰지를 판단
* 수행하는 계산
$$
i = \sigma(x_t W_x^{(i)} + h_{t-1} W_h^{(i)} + b^{(i)})
$$
* i와 g의 원소별 곱 결과를 기억 셀에 추가

#### LSTM의 기울기 흐름
* 기억 셀 c의 역전파에 주목하면 기울기 소실을 없앰
* 기억 셀의 역전파에서는 +와 x 노드만을 지남  
    * \+ 노드에서는 기울기 변화는 일어나지 않음
    * x 노드에서는 원소별 곱을 계산
        * 매번 다른 게이트 값을 이용해 원소별 곱을 계산  
    => 기울기 소실이 일어나기 어려움
    * x 노드의 계산은 forget 게이트가 제어

### LSTM 구현

* LSTM에서 수행하는 계산을 정리한 수식
$$
f = \sigma(x_t W_x^{(f)} + h_{t-1} W_h^{(f)} + b^{(f)})
$$
$$
g = \tanh(x_t W_x^{(g)} + h_{t-1} W_h^{(g)} + b^{(g)})
$$
$$
i = \sigma(x_t W_x^{(i)} + h_{t-1} W_h^{(i)} + b^{(i)})
$$
$$
o = \sigma(x_t W_x^{(o)} + h_{t-1} W_h^{(o)} + b^{(o)})
$$
$$
c_t = f \odot c_{t-1} + g \odot i \\
$$
$$
h_t = o \odot \tanh(c_t)
$$

* 주목할 부분은 네 수식에 포함된 아핀 변환
    * 아핀 변환: 행렬 변환과 평행 이동을 결합한 형태
* 각 식의 가중치들을 모아 4개의 식을 단 한 번의 아핀 변환으로 계산
$$
x_t [W_x^{(f)} \ W_x^{(g)}\ W_x^{(i)}\ W_x^{(o)}] + h_{t-1} [W_x^{(f)}\ W_x^{(g)}\ W_x^{(i)}\ W_x^{(o)}] + [b^{(f)} \ b^{(g)} \ b^{(i)} \ b^{(o)}]
$$
$$
\rightarrow x_t W_x + h_{t-1} W_h + b
$$

In [5]:
import numpy as np

In [6]:
class LSTM:
    def __init__(self, Wx, Wh, b):
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.cache = None

* 초기화 인수는 가중치 매개변수인 Wx, Wh, 편향 b
* params에 할당 -> 이에 대응하는 형태로 기울기도 초기화
* cache는 순전파 때 중간 결과를 보관했다가 역전파 계산에 사용하려는 용도의 인스턴스 변수

In [9]:
def forward(self, x, h_prev, c_prev):
    Wx, Wh, b = self.params
    N, H = h_prev.shape

    A = np.matmul(x, Wx) + np.matmul(h_prev, Wh) + b

    # slice
    f = A[:, :H]
    g = A[:, H:2 * H]
    i = A[:, 2 * H:3 * H]
    o = A[:, 3 * H:]

    f = sigmoid(f)
    g = np.tanh(g)
    i = sigmoid(i)
    o = sigmoid(o)

    c_next = f * c_prev + g * i
    h_next = o * np.tanh(c_next)

    self.cache = (x, h_prev, c_prev, i, f, g, o, c_next)
    return h_next, c_next


* 미니배치 수 N, 입력 데이터의 차원 수 D, 기억 셀과 은닉 상태의 차원 수 H
* A에는 아핀 변환 결과가 저장, 슬라이스 형태로 데이터를 꺼냄, 꺼낸 데이터를 다음 연산 노드에 분배

In [12]:
dA = np.hstack((df, dg, di, do))

* slice 노드의 역전파

#### Time LSTM 구현
* T개분의 시계열 데이터를 한번에 처리하는 계층

In [13]:
class TimeLSTM:
    def __init__(self, Wx, Wh, b, stateful=False):
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.layers = None
        self.h, self.c = None, None
        self.dh = None
        self.stateful = stateful

    def forward(self, xs):
        Wx, Wh, b = self.params
        N, T, D = xs.shape
        H = Wh.shape[0]

        self.layers = []
        hs = np.empty((N, T, H), dtype='f')

        if not self.stateful or self.h is None:
            self.h = np.zeros((N, H), dtype='f')
        if not self.stateful or self.c is None:
            self.c = np.zeros((N, H), dtype='f')
        
        for t in range(T):
            layer = LSTM(*self.params)
            self.h, self.c = layer.forward(xs[:, t, :], self.h, self.c)
            hs[:, t, :] = self.h

            self.layers.append(layer)
        
        return hs
    
    def backward(self, dhs):
        Wx, Wh, b = self.params
        N, T, H = dhs.shape
        D = Wx.shape[0]

        dxs = np.empty((N, T, D), dtype='f')
        dh, dc = 0, 0

        grads = [0, 0, 0]
        for t in reversed(range(T)):
            layer = self.layers[t]
            dx, dh, dc = layer.backward(dhs[:, t, :] + dh, dc)
            dxs[:, t, :] = dx
            for i, grad in enumerate(layer.grads):
                grads[i] += grad

        for i, grad in enumerate(grads):
            self.grads[i][...] = grad
            self.dh = dh
            return dxs
    
    def set_state(self, h, c=None):
        self.h, self.c = h, c
    
    def reset_state(self):
        self.h, self.c = None, None

* TimeLSTM 클래스 구현은 Time RNN 구현과 유사함
* stateful로 상태를 유지할지를 지정함

### LSTM을 사용한 언어 모델

In [ ]:
import sys
sys.path.append('..')
from common.time_layers import *
import pickle

class Rnnlm:
    def __init__(self, vocab_size=10000, wordvec_size=100, hidden_size=100):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        # 가중치 초기화
        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b = np.zeros(4 * H).astype('f')
        affine_W = (rn(H, V) / np.sqrt(H)).astype('f')
        affine_b = np.zeros(V).astype('f')

        # 계층 생성
        self.layers = [
            TimeEmbedding(embed_W),
            TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=True),
            TimeAffine(affine_W, affine_b)
        ]
        self.loss_layer = TimeSoftmaxWithLoss()
        self.lstm_layer = self.layers[1]

        # 모든 가중치와 기울기를 리스트에 모은다.
        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def predict(self, xs):
        for layer in self.layers:
            xs = layer.forward(xs)
        return xs

    def forward(self, xs, ts):
        score = self.predict(xs)
        loss = self.loss_layer.forward(score, ts)
        return loss

    def backward(self, dout=1):
        dout = self.loss_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def reset_state(self):
        self.lstm_layer.reset_state()

    def save_params(self, file_name='Rnnlm.pkl'):
        with open(file_name, 'wb') as f:
            pickle.dump(self.params, f)
    
    def load_params(self, file_name='Rnnlm.pkl'):
        with open(file_name, 'rb') as f:
            self.params = pickle.load(f)

* Softmax 계층 직전까지를 처리하는 predict() 메서드 -> 문장 생성에 사용

In [ ]:
import sys
sys.path.append('..')
from common.optimizer import SGD
from common.trainer import RnnlmTrainer
from common.util import eval_perplexity
from dataset import ptb
from rnnlm import Rnnlm


# 하이퍼파라미터 설정
batch_size = 20
wordvec_size = 100
hidden_size = 100  # RNN의 은닉 상태 벡터의 원소 수
time_size = 35     # RNN을 펼치는 크기
lr = 20.0
max_epoch = 4
max_grad = 0.25

# 학습 데이터 읽기
corpus, word_to_id, id_to_word = ptb.load_data('train')
corpus_test, _, _ = ptb.load_data('test')
vocab_size = len(word_to_id)
xs = corpus[:-1]
ts = corpus[1:]

# 모델 생성
model = Rnnlm(vocab_size, wordvec_size, hidden_size)
optimizer = SGD(lr)
trainer = RnnlmTrainer(model, optimizer)

# 기울기 클리핑을 적용하여 학습
trainer.fit(xs, ts, max_epoch, batch_size, time_size, max_grad,
            eval_interval=20)
trainer.plot(ylim=(0, 500))

# 테스트 데이터로 평가
model.reset_state()
ppl_test = eval_perplexity(model, corpus_test)
print('테스트 퍼플렉서티: ', ppl_test)

# 매개변수 저장
model.save_params()

### RNNLM 추가 개선

#### LSTM 계층 다층화
* RNNLM으로 정확한 모델을 만들고자 한다면 많은 경우 LSTM 계층을 깊게 쌓아 효과를 볼 수 있음
* 층 수는 하이퍼파라미터와 관한 문제

#### 드롭아웃에 의한 과적합 억제
* 층을 깊게 쌓음으로써 표현력이 풍부한 모델을 만들 수 있음
-> 종종 과적합을 일으킴
* RNN은 일반적인 피드포워드 신경망보다 쉽게 과적합을 일으킴

* 과적합은 훈련 데이터의 양 늘리기 및 모델의 복잡도를 줄이는 것이 억제하는 보통의 방법
* 모델의 복잡도에 페널티를 주는 정규화도 효과적임
    * 드롭아웃처럼 훈련 시 계층 내의 뉴런 몇 개를 무작위로 무시하고 학습하는 방법도 정규화라고 할 수 있음

* 일반적인 드롭아웃은 시간 방향에는 적합하지 않음
    * 시계열 방향으로 드롭아웃을 넣어버리면 시간이 흐름에 따라 정보가 사라질 수 있음
    * 흐르는 시간에 비례해 드롭아웃에 의한 노이즈가 축적됨
* 변형 드롭아웃은 깊이 방향은 물론, 시간 방향에도 이용할 수 있음
* 같은 계층에 속한 드롭아웃들은 같은 마스크를 공유함
    * 마스크: 데이터의 통과/차단을 결정하는 이진 형태의 무작위 패턴
* 같은 계층의 드롭아웃끼리 마스크를 공유함으로써 마스크가 고정됨  
    -> 정보를 잃게 되는 방법도 고정되므로 정보가 지수적으로 손실되는 사태를 피할 수 있음

#### 가중치 공유
* 가중치를 연결한다지만 가중치를 공유하는 효과를 줌
* Embedding 계층의 가중치와 Affine 계층의 가중치를 연결하는 기법이 가중치 공유
* 두 계층이 가중치를 공유함으로써 학습하는 매개변수의 수가 크게 줄어드는 동시에 정확도도 향상됨

* 어휘 수를 V, LSTM 은닉 상태의 차원 수를 H
* Embedding 계층의 가중치는 형상이 V * H이며, Affine 계층의 가중치 형상은 H * V가 됨
* 가중치 공유를 적용하려면 Embedding 가중치를 전치해 Affine 계층의 가중치로 설정하기만 하면 됨

#### 개선된 RNNLM 구현
* 개선점
    * LSTM 계층의 다층화
    * 드롭아웃 사용(깊이 방향)
    * 가중치 공유

In [ ]:
import sys
sys.path.append('..')
from common.time_layers import *
from common.np import *
from common.base_model import BaseModel

class BetterRnnlm(BaseModel):
    def __init__(self, vocab_size=10000, wordvec_size=650,
                 hidden_size=650, dropout_ratio=0.5):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx1 = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh1 = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b1 = np.zeros(4 * H).astype('f')
        lstm_Wx2 = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_Wh2 = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b2 = np.zeros(4 * H).astype('f')
        affine_b = np.zeros(V).astype('f')

        self.layers = [
            TimeEmbedding(embed_W),
            TimeDropout(dropout_ratio),
            TimeLSTM(lstm_Wx1, lstm_Wh1, lstm_b1, stateful=True),
            TimeDropout(dropout_ratio),
            TimeLSTM(lstm_Wx2, lstm_Wh2, lstm_b2, stateful=True),
            TimeDropout(dropout_ratio),
            TimeAffine(embed_W.T, affine_b)  # weight tying!!
        ]
        self.loss_layer = TimeSoftmaxWithLoss()
        self.lstm_layers = [self.layers[2], self.layers[4]]
        self.drop_layers = [self.layers[1], self.layers[3], self.layers[5]]

        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def predict(self, xs, train_flg=False):
        for layer in self.drop_layers:
            layer.train_flg = train_flg

        for layer in self.layers:
            xs = layer.forward(xs)
        return xs

    def forward(self, xs, ts, train_flg=True):
        score = self.predict(xs, train_flg)
        loss = self.loss_layer.forward(score, ts)
        return loss

    def backward(self, dout=1):
        dout = self.loss_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def reset_state(self):
        for layer in self.lstm_layers:
            layer.reset_state()

* TimeLSTM 계층을 2개 겹치고 사이사이에 TimeDropout 계층을 사용
* TimeEmbedding 계층과 TimeAffine 계층에서 가중치를 공유

In [ ]:
import sys
sys.path.append('..')
from common import config
# GPU에서 실행하려면 아래 주석을 해제하세요(CuPy 필요).
# ==============================================
# config.GPU = True
# ==============================================
from common.optimizer import SGD
from common.trainer import RnnlmTrainer
from common.util import eval_perplexity, to_gpu
from dataset import ptb
from better_rnnlm import BetterRnnlm


# 하이퍼파라미터 설정
batch_size = 20
wordvec_size = 650
hidden_size = 650
time_size = 35
lr = 20.0
max_epoch = 40
max_grad = 0.25
dropout = 0.5

# 학습 데이터 읽기
corpus, word_to_id, id_to_word = ptb.load_data('train')
corpus_val, _, _ = ptb.load_data('val')
corpus_test, _, _ = ptb.load_data('test')

if config.GPU:
    corpus = to_gpu(corpus)
    corpus_val = to_gpu(corpus_val)
    corpus_test = to_gpu(corpus_test)

vocab_size = len(word_to_id)
xs = corpus[:-1]
ts = corpus[1:]

model = BetterRnnlm(vocab_size, wordvec_size, hidden_size, dropout)
optimizer = SGD(lr)
trainer = RnnlmTrainer(model, optimizer)

best_ppl = float('inf')
for epoch in range(max_epoch):
    trainer.fit(xs, ts, max_epoch=1, batch_size=batch_size,
                time_size=time_size, max_grad=max_grad)

    model.reset_state()
    ppl = eval_perplexity(model, corpus_val)
    print('검증 퍼플렉서티: ', ppl)

    if best_ppl > ppl:
        best_ppl = ppl
        model.save_params()
    else:
        lr /= 4.0
        optimizer.lr = lr

    model.reset_state()
    print('-' * 50)


# 테스트 데이터로 평가
model.reset_state()
ppl_test = eval_perplexity(model, corpus_test)
print('테스트 퍼플렉서티: ', ppl_test)

#### 첨단 연구로
* 계속된 새로운 기법이 나타나면서 퍼플렉서티도 점차 내려감
* 첨단 모델과의 공통점
    * 첨단 모델에서도 다층 LSTM을 사용
    * 드롭아웃 기반의 정규화
    * 가중치 공유